# ML 3

## Questions

leave-one-out - медод кросс-валидации, когда валидационный фолд состоит только из одного элемента.
Плюсы:
- Позволяет обучаться на максимальном количестве данных
- Хорошо работает на небольших датасетах

Минусы:
- Вычислительно очень сложен

Grid Search:
- Простой перепор комбинации всех параметров по заданной сетке

Randomized Grid Search:
- Случайный перебор в заданных границах по заданному распределению

Bayesian optimization:
- Побочная модель связи между результатом основной модели и гиперпараметрами
- Оптимизация гиперпараметров по побочной модели
- Может экономить очень много времени для моделей с высоким временем обучения и возможностью ранней остановки (нейросети, бустинги)
- Пример - Optuna

Методы отбора фичей можно грубо поделить на:
- методы без учителя(таргет не используется)
- с учителем (таргет используется)


Корреляция Пирсона:
- Попытка измерить, насколько хорошо через разброс точек на графике можно провести прямую линию
- Просто и очень быстро
- Только линейные зависмости
- Только для непрерывных признаков

Chi2:
- Основан на частотах, считает ожидаемую частоту при полной независисоти и фактическую
- Просто и очень быстро
- Только линейные зависимости
- Только для дисретных(категориальных) признаков

Lasso:
- Применение штрафа L1 к весам внутри модели, что может занулить веса слабых признаков

Permutation significance(Вообще-то permutation importance)
- Идея - "сломать" связь модели с каким-то признаком
- Обучаем модель, потом на валидации шафлим признак, смотрим как изменится метрика

## Intro

In [1]:
import pandas as pd
import numpy as np
import optuna

from scipy.stats import uniform
from collections import Counter


from sklearn.preprocessing import OrdinalEncoder, MultiLabelBinarizer, StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import Lasso, ElasticNet

import shap

c:\Users\Philka\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
data = pd.read_json('../datasets/train.json')

In [3]:
data['created'] = pd.to_datetime(data['created'])

In [4]:
upper_price_limit = np.percentile(data.price.values, 99)
lower_pirce_limit = np.percentile(data.price.values, 1)
data = data[data['price'] < upper_price_limit]
data = data[data['price'] > lower_pirce_limit]

In [5]:
ord_encoder = OrdinalEncoder()
ord_encoder.fit(data[['interest_level']])
data['interest_level'] = ord_encoder.transform(data[['interest_level']])

In [6]:
def clean_list(values: list):
    if isinstance(values, list):
        return [item.replace('[', '')
             .replace(']', '')
             .replace("'", '')
             .replace('"', '') 
             .replace(' ', '') for item in values]
    return values

In [7]:
data['features'] = data['features'].apply(clean_list)

In [8]:
list_of_features = data['features'].sum()
count_features = Counter(list_of_features)
most_common_features = [feature[0] for feature in count_features.most_common(20)]
count_features.most_common(20)

[('Elevator', 25375),
 ('HardwoodFloors', 23146),
 ('CatsAllowed', 23135),
 ('DogsAllowed', 21652),
 ('Doorman', 20479),
 ('Dishwasher', 20081),
 ('NoFee', 17793),
 ('LaundryinBuilding', 16082),
 ('FitnessCenter', 12989),
 ('Pre-War', 8971),
 ('LaundryinUnit', 8437),
 ('RoofDeck', 6417),
 ('OutdoorSpace', 5132),
 ('DiningRoom', 4890),
 ('HighSpeedInternet', 4223),
 ('Balcony', 2898),
 ('SwimmingPool', 2643),
 ('LaundryInBuilding', 2564),
 ('NewConstruction', 2504),
 ('Terrace', 2177)]

In [9]:
features_col = data.iloc[:, 6]
mlb = MultiLabelBinarizer()
encoded_features = mlb.fit_transform(features_col)
features_df = pd.DataFrame(encoded_features, columns=mlb.classes_, index=data.index)

for feature in most_common_features:
    if feature in features_df.columns:
        data[feature] = features_df[feature]
    else:
        data[feature] = 0

In [10]:
columns_to_process = ['bathrooms', 'bedrooms', 'interest_level', 'created', 'price'] + most_common_features

In [11]:
data = data[columns_to_process]

## Split methods 

In [12]:
def my_train_test_split(data, test_size):
    inds = np.random.permutation(data.shape[0])
    data = data.iloc[inds].reset_index(drop=True)
    test = data.iloc[:int(data.shape[0] * test_size),:]
    train = data.iloc[int(data.shape[0] * test_size):,:]
    return train, test

In [13]:
train, test = my_train_test_split(data.iloc[:133,], 0.6)
display(data.shape, train.shape, test.shape)

(48343, 25)

(54, 25)

(79, 25)

In [14]:
def my_train_val_test_split(data, test_size, validation_size, random_state=None):
    generator = np.random.default_rng(random_state)
    inds = generator.permutation(data.shape[0])
    data = data.iloc[inds].reset_index(drop=True)
    test = data.iloc[:int(data.shape[0] * test_size),:] 
    val = data.iloc[int(data.shape[0] * test_size): int(data.shape[0] * (test_size + validation_size)),:] 
    train = data.iloc[int(data.shape[0] * (test_size + validation_size)):,:] 
    return train, val, test


In [15]:
train, test, test = my_train_val_test_split(data.iloc[:133,], 0.3, 0.3)
display(data.shape, train.shape, test.shape, test.shape)

(48343, 25)

(54, 25)

(39, 25)

(39, 25)

In [16]:
def my_train_test_date_split(data: pd.DataFrame, test_size, date):
    data = data.sort_values(date, ascending=True, ignore_index=True)
    train = data.iloc[:int(data.shape[0] * (1 - test_size)),:]
    test = data.iloc[int(data.shape[0] * (1 - test_size)):,:]
    return train, test

In [17]:
train, test = my_train_test_date_split(data, 0.4, 'created')
display(data.shape, train.shape, test.shape)

(48343, 25)

(29005, 25)

(19338, 25)

In [18]:
def my_train_test_vale_date_split(data: pd.DataFrame, test_size, validation_size, date_split):
    data = data.sort_values(date_split, ascending=True, ignore_index=True)
    train = data.iloc[:int(data.shape[0] * (1 - test_size - validation_size)),:]
    val = data.iloc[int(data.shape[0] * (1 - test_size - validation_size)): int(data.shape[0] * (1 - test_size)),:]
    test = data.iloc[int(data.shape[0] * (1 - test_size)):,:]
    return train, val, test

In [19]:
train, test, test = my_train_test_vale_date_split(data.iloc[:133,], 0.3, 0.3, 'created')
display(data.shape, train.shape, test.shape, test.shape)

(48343, 25)

(53, 25)

(40, 25)

(40, 25)

# Cross-validation methods

In [20]:
class my_KFold():
    def __init__(self, n_splits=5, shuffle=False, random_state=None):
        self.random_generator = np.random.default_rng(random_state)
        self.n_splits = n_splits
        self.shuffle = shuffle

    def split(self, data):
        if self.shuffle:
            indxs = self.random_generator.permutation(len(data))
        else:
            indxs = np.arange(len(data))

        fold_sizes = (len(data) // self.n_splits) * np.ones(self.n_splits, dtype=int)
        fold_sizes[:len(data) % self.n_splits] += 1

        current = 0
        for fold_size in fold_sizes:
            start, stop = current, current + fold_size
            test_indices = indxs[start:stop]
            train_indices = np.concatenate([indxs[:start], indxs[stop:]])
            yield train_indices, test_indices 
            current = stop

In [21]:
class my_GroupKFold():
    def __init__(self, n_splits=5, shuffle=False, random_state=None):
        self.random_generator = np.random.default_rng(random_state)
        self.n_splits = n_splits
        self.shuffle = shuffle

    def split(self, data, group_field):
        groups = data[group_field].unique()

        if len(groups) < self.n_splits:
            raise ValueError('Количество групп должно быть меньше или равно количеству фолдов')
        
        if self.shuffle:
            indxs = self.random_generator.permutation(len(groups))
            groups = groups[indxs]

        groups_per_fold = len(groups) // self.n_splits * np.ones(self.n_splits, dtype=int)
        groups_per_fold[:len(groups) % self.n_splits] += 1

        current = 0
        for n_of_groups in groups_per_fold:
            start, stop = current, current + n_of_groups
            test_groups = groups[start:stop]

            test_mask = data[group_field].isin(test_groups)
            test_indices = np.where(test_mask)[0]
            train_indices = np.where(~test_mask)[0]
    
            yield train_indices, test_indices
            current = stop



In [22]:
cv = my_GroupKFold(n_splits=3, shuffle=True, random_state=42)

for train_idx, test_idx in cv.split(data, 'bathrooms'):
    print(data.iloc[train_idx].shape)
    print(data.iloc[test_idx].shape)


(9167, 25)
(39176, 25)
(39823, 25)
(8520, 25)
(47696, 25)
(647, 25)


In [23]:
class my_StratifiedKFold:
    def __init__(self, n_splits=5, random_state=None):
        self.n_splits = n_splits
        self.random_generator = np.random.default_rng(random_state)

    def split(self, data, stratify_field):
        y = data[stratify_field].to_numpy()
        n = len(y)

        class_folds = {}
        for cls in np.unique(y):
            cls_idx = np.flatnonzero(y == cls)           
            cls_idx = self.random_generator.permutation(cls_idx)
            class_folds[cls] = np.array_split(cls_idx, self.n_splits)

        for k in range(self.n_splits):
            test_parts = [class_folds[cls][k] for cls in class_folds]
            test = np.concatenate(test_parts) if test_parts else np.empty(0, dtype=int)

            if test.size:
                test = test[self.random_generator.permutation(test.size)]

            train_mask = np.ones(n, dtype=bool)
            train_mask[test] = False
            train = np.flatnonzero(train_mask)

            yield train, test

In [24]:
cv = my_StratifiedKFold(n_splits=3, random_state=None)

for train_idx, test_idx in cv.split(data, 'bathrooms'):
    display(data.iloc[test_idx]['bathrooms'].value_counts(normalize=True))

bathrooms
1.0     0.804752
2.0     0.156595
1.5     0.013339
3.0     0.011664
0.0     0.006266
2.5     0.004777
4.0     0.001737
3.5     0.000682
4.5     0.000062
5.0     0.000062
10.0    0.000062
Name: proportion, dtype: float64

bathrooms
1.0    0.804902
2.0    0.156624
1.5    0.013342
3.0    0.011666
0.0    0.006267
2.5    0.004778
4.0    0.001738
3.5    0.000683
Name: proportion, dtype: float64

bathrooms
1.0    0.805152
2.0    0.156611
1.5    0.013346
3.0    0.011670
0.0    0.006207
2.5    0.004718
4.0    0.001676
3.5    0.000621
Name: proportion, dtype: float64

In [25]:
class my_TimeSeriesSplit():
    def __init__(self, n_splits=5):
        self.n_splits = n_splits + 1

    def split(self, data: pd.DataFrame, date_field):
        indxs = np.argsort(np.asarray(data[date_field]), kind='mergesort')
        n_samples = len(data) // self.n_splits
        
        folds = n_samples * np.ones(self.n_splits, dtype=int)
        folds[:len(data) % self.n_splits] += 1

        current = folds[0]
        for i in range(1, self.n_splits):
            start, stop = current, current + folds[i]
            test = indxs[start:stop]
            train =  indxs[:start]
            yield train, test
            current = stop

In [26]:
cv = my_TimeSeriesSplit(n_splits=5)

for train_idx, test_idx  in cv.split(data, 'created'):
    display(data.iloc[train_idx]['created'], data.iloc[test_idx]['created'])
    break


111817   2016-04-01 22:12:41
117995   2016-04-01 22:56:00
114617   2016-04-01 22:57:15
117474   2016-04-01 23:26:07
103891   2016-04-02 00:48:13
                 ...        
113976   2016-04-16 01:29:22
110902   2016-04-16 01:29:25
96640    2016-04-16 01:29:43
111339   2016-04-16 01:29:59
89730    2016-04-16 01:30:37
Name: created, Length: 8058, dtype: datetime64[ns]

85567    2016-04-16 01:31:06
89876    2016-04-16 01:31:06
98185    2016-04-16 01:31:48
110527   2016-04-16 01:38:03
110706   2016-04-16 01:38:09
                 ...        
49641    2016-05-02 01:13:45
66272    2016-05-02 01:13:47
43289    2016-05-02 01:13:57
62012    2016-05-02 01:13:58
67518    2016-05-02 01:14:08
Name: created, Length: 8057, dtype: datetime64[ns]

## Cross-validation comparison

Вообще метод кросс-валидации нужно выбирать исходя из данных и задачи, здесь (прогноз Price) хватит и обычного KFolds, можно сделать распределение равнее по фолдам, если разбить таргет по бинам и использвать Stratified K-fold.

In [27]:
data['bins'] = pd.qcut(data['price'], 5, labels=False)

In [28]:
data['bins']

4         1
6         3
9         3
10        2
15        1
         ..
124000    1
124002    1
124004    0
124008    3
124009    3
Name: bins, Length: 48343, dtype: int64

In [29]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=21)

for train_idx, test_idx in cv.split(data, data['bins']):
    print(data.iloc[train_idx].shape)
    print(data.iloc[test_idx].shape)

(38674, 26)
(9669, 26)
(38674, 26)
(9669, 26)
(38674, 26)
(9669, 26)
(38675, 26)
(9668, 26)
(38675, 26)
(9668, 26)


In [30]:
data = data.drop('bins', axis=1)

## Feature Selection

In [31]:
data = data.drop('created', axis=1)

In [32]:
result_MAE = pd.DataFrame(columns=['model', 'train', 'val', 'test'])
result_RMSE = pd.DataFrame(columns=['model', 'train', 'val', 'test'])
result_R_2 = pd.DataFrame(columns=['model', 'train', 'val', 'test'])

In [33]:
train, val, test = my_train_val_test_split(data, validation_size=0.2, test_size=0.2)

In [34]:
lasso_regr = Lasso()

In [35]:
X_train = train.drop(['price'], axis=1)
y_train = train['price']

X_val = val.drop(['price'], axis=1)
y_val = val['price']

X_test = val.drop(['price'], axis=1)
y_test = val['price']

In [36]:
scaler = StandardScaler()

In [37]:
X_train[scaler.get_feature_names_out()] = scaler.fit_transform(X_train)
X_val[scaler.get_feature_names_out()] = scaler.transform(X_val)
X_test[scaler.get_feature_names_out()] = scaler.transform(X_test)

In [38]:
lasso_regr.fit(X_train, y_train)

,alpha,1.0
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [39]:
def check_model_metrics(model, model_name, X_train, X_test, y_train, y_test, X_val, y_val,
                       result_R_2, result_RMSE, result_MAE):
    # Предсказания и метрики для тренировочной и тестовой выборок
    train_preds = model.predict(X_train)
    train_r2 = r2_score(y_train, train_preds)
    train_rmse = root_mean_squared_error(y_train, train_preds)
    train_mae = mean_absolute_error(y_train, train_preds)
    
    test_preds = model.predict(X_test)
    test_r2 = r2_score(y_test, test_preds)
    test_rmse = root_mean_squared_error(y_test, test_preds)
    test_mae = mean_absolute_error(y_test, test_preds)
        
    # Расчет метрик для валидационной выборки, если данные предоставлены
    val_preds = model.predict(X_val)
    val_r2 = r2_score(y_val, val_preds)
    val_rmse = root_mean_squared_error(y_val, val_preds)
    val_mae = mean_absolute_error(y_val, val_preds)
    
    # Добавление результатов в DataFrame
    new_row_r2 = {'model': model_name, 'train': train_r2, 'test': test_r2, 'val': val_r2}
    new_row_rmse = {'model': model_name, 'train': train_rmse, 'test': test_rmse, 'val': val_rmse}
    new_row_mae = {'model': model_name, 'train': train_mae, 'test': test_mae, 'val': val_mae}
    
    result_R_2 = pd.concat([result_R_2, pd.DataFrame([new_row_r2])], ignore_index=True)
    result_RMSE = pd.concat([result_RMSE, pd.DataFrame([new_row_rmse])], ignore_index=True)
    result_MAE = pd.concat([result_MAE, pd.DataFrame([new_row_mae])], ignore_index=True)

    return result_R_2, result_RMSE, result_MAE

In [40]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(lasso_regr, 
                                                          'Lasso Standart Scaler', 
                                                          X_train, 
                                                          X_test, 
                                                          y_train, 
                                                          y_test,
                                                          X_val, 
                                                          y_val,
                                                          result_R_2, result_RMSE, result_MAE)

C:\Users\Philka\AppData\Local\Temp\ipykernel_26216\3196882686.py:25: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result_R_2 = pd.concat([result_R_2, pd.DataFrame([new_row_r2])], ignore_index=True)
C:\Users\Philka\AppData\Local\Temp\ipykernel_26216\3196882686.py:26: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  result_RMSE = pd.concat([result_RMSE, pd.DataFrame([new_row_rmse])], ignore_index=True)
C:\Users\Philka\AppData\Local\Temp\ipykernel_26216\3196882686.py:27: FutureWarning: The behavior o

### Simple_coefs_imp

In [41]:
def find_importance(model, feature_names):
    importance = np.abs(model.coef_)
    feat_imp = pd.Series(importance, index=feature_names).sort_values(ascending=False)
    return feat_imp.head(10)

In [42]:
imp_results = find_importance(lasso_regr, X_train.columns)
imp_results

bathrooms            718.172340
bedrooms             503.327041
Doorman              296.537764
LaundryinUnit        175.312568
Elevator             120.801050
LaundryinBuilding    104.031566
FitnessCenter         97.815549
NoFee                 75.872414
LaundryInBuilding     75.203237
interest_level        68.703565
dtype: float64

In [43]:
top_10_simple = imp_results.index

In [44]:
X_train_simple = X_train[top_10_simple]
X_val_simple = X_val[top_10_simple]
X_test_simple = X_test[top_10_simple]

In [45]:
lasso_regr_simple = Lasso()

In [46]:
lasso_regr_simple.fit(X_train_simple, y_train)

,alpha,1.0
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [47]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(lasso_regr_simple, 
                                                          'Lasso Standart Scaler top10', 
                                                          X_train_simple, 
                                                          X_test_simple, 
                                                          y_train, 
                                                          y_test,
                                                          X_val_simple, 
                                                          y_val,
                                                          result_R_2, result_RMSE, result_MAE)

## Permutation 

In [48]:
def perm_importance(model, X_test, y_test):
    
    perm_result = permutation_importance(
                                        model, 
                                        X_test, 
                                        y_test,
                                        n_repeats=10,  
                                        random_state=42
                                        )
    
    mean_abs = perm_result.importances_mean
    
    feat_imp = pd.Series(mean_abs, index=X_test.columns).sort_values(ascending=False)
    return feat_imp.head(10)
    

In [49]:
permute_values = perm_importance(lasso_regr, X_test, y_test)
permute_values

bathrooms            0.402671
bedrooms             0.196172
Doorman              0.069435
LaundryinUnit        0.023294
Elevator             0.010843
LaundryinBuilding    0.009201
FitnessCenter        0.008703
LaundryInBuilding    0.004800
HardwoodFloors       0.004404
NoFee                0.003947
dtype: float64

In [50]:
top_10_permute = permute_values.index

In [51]:
X_train_permute = X_train[top_10_permute]
X_val_permute = X_val[top_10_permute]
X_test_permute = X_test[top_10_permute]

In [52]:
lasso_regr_permute = Lasso()
lasso_regr_permute.fit(X_train_permute, y_train)

,alpha,1.0
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [53]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(lasso_regr_permute, 
                                                          'Lasso Standart Scaler Perm Imp', 
                                                          X_train_permute, 
                                                          X_test_permute, 
                                                          y_train, 
                                                          y_test,
                                                          X_val_permute, 
                                                          y_val,
                                                          result_R_2, result_RMSE, result_MAE)

## Shap

In [54]:
def find_shap(model, X_train, X_test):

    explainer = shap.LinearExplainer(model, X_train)
    shap_values = explainer.shap_values(X_test)
    mean_abs_shap = np.mean(np.abs(shap_values), axis=0)

    feat_imp = pd.Series(mean_abs_shap, index=X_train.columns).sort_values(ascending=False)
    return feat_imp.head(10)

In [55]:
shap_values = find_shap(lasso_regr, X_train, X_test)

In [56]:
shap_values

bathrooms            519.679675
bedrooms             426.813236
Doorman              296.415579
LaundryinUnit        133.388276
Elevator             120.064230
LaundryinBuilding     92.618492
FitnessCenter         88.502888
NoFee                 74.040635
HardwoodFloors        65.766047
Dishwasher            63.233665
dtype: float64

In [57]:
shap_features = shap_values.index

In [58]:
lasso_shap = Lasso()

In [59]:
X_train_shap = X_train[shap_features]
X_val_shap = X_val[shap_features]
X_test_shap = X_test[shap_features]

In [60]:
lasso_shap.fit(X_train_shap, y_train)

,alpha,1.0
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


In [61]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(lasso_shap, 
                                                          'Lasso Standart Scaler Shap', 
                                                          X_train_shap, 
                                                          X_test_shap, 
                                                          y_train, 
                                                          y_test,
                                                          X_val_shap, 
                                                          y_val,
                                                          result_R_2, result_RMSE, result_MAE)

In [62]:
display(result_MAE, result_RMSE, result_R_2)

,model,train,val,test
0,Lasso Standart Scaler,705.165992,709.243433,709.243433
1,Lasso Standart Scaler top10,711.763983,717.353878,717.353878
2,Lasso Standart Scaler Perm Imp,713.158844,716.185760,716.185760
3,Lasso Standart Scaler Shap,712.924466,716.413476,716.413476


,model,train,val,test
0,Lasso Standart Scaler,1015.637572,1043.037205,1043.037205
1,Lasso Standart Scaler top10,1022.558048,1052.114534,1052.114534
2,Lasso Standart Scaler Perm Imp,1024.157766,1050.683104,1050.683104
3,Lasso Standart Scaler Shap,1025.298559,1052.731072,1052.731072


,model,train,val,test
0,Lasso Standart Scaler,0.586885,0.571805,0.571805
1,Lasso Standart Scaler top10,0.581236,0.564320,0.564320
2,Lasso Standart Scaler Perm Imp,0.579924,0.565504,0.565504
3,Lasso Standart Scaler Shap,0.578988,0.563809,0.563809


## Hyperparameter optimization

In [63]:
class ElasticNetGridSearch():
    def __init__(self):
        pass

    def fit(self, X_train, y_train, X_val, y_val, l1, l2):
        self.best_model = None
        self.best_loss = np.inf
        self.best_params = None
        for l1_value in l1:
            for l2_value in l2:
                elastic_model = ElasticNet(l1_ratio=l1_value, alpha=l2_value)
                elastic_model.fit(X_train, y_train)

                val_los = root_mean_squared_error(y_val, elastic_model.predict(X_val))

                if val_los < self.best_loss:
                    self.best_loss = val_los
                    self.best_model = elastic_model
                    self.best_params = (l1_value, l2_value)
        return self

In [64]:
elastic_grid = ElasticNetGridSearch()

l1_range = np.arange(0.1, 1, 0.1)
l2_range = np.arange(0.1, 2, 0.25)

elastic_grid.fit(X_train, y_train, X_val, y_val, l1_range, l2_range)

In [65]:
grid_model = elastic_grid.best_model
elastic_grid.best_loss

1042.7298792233132

In [66]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(grid_model, 
                                                          'Elastic Standart Scaler Grid', 
                                                          X_train, 
                                                          X_test, 
                                                          y_train, 
                                                          y_test,
                                                          X_val, 
                                                          y_val,
                                                          result_R_2, result_RMSE, result_MAE)

In [67]:
class ElasticNetRandomizedSearch():
    def __init__(self):
        pass
    
    def fit(self, X_train, y_train, X_val, y_val, l1_dist, l2_dist, n_trials):
        self.best_model = None
        self.best_loss = np.inf
        self.best_params = None
        n_obj = int(np.sqrt(n_trials)) + 1
        l1 = l1_dist.rvs(size=n_obj)
        l2 = l2_dist.rvs(size=n_obj, scale=2)
        for l1_value in l1:
            for l2_value in l2:
                elastic_model = ElasticNet(l1_ratio=l1_value, alpha=l2_value)
                elastic_model.fit(X_train, y_train)

                val_los = root_mean_squared_error(y_val, elastic_model.predict(X_val))

                if val_los < self.best_loss:
                    self.best_loss = val_los
                    self.best_model = elastic_model
                    self.best_params = (l1_value, l2_value)
        return self


In [68]:
elastic_random = ElasticNetRandomizedSearch()
elastic_random.fit(X_train, y_train, X_val, y_val, uniform, uniform, n_trials=100)

In [69]:
random_model = elastic_random.best_model
elastic_random.best_loss

1042.7097434311024

In [70]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(random_model, 
                                                          'Elastic Standart Scaler Random', 
                                                          X_train, 
                                                          X_test, 
                                                          y_train, 
                                                          y_test,
                                                          X_val, 
                                                          y_val,
                                                          result_R_2, result_RMSE, result_MAE)

## Optuna

In [71]:
class ElasticNetOptuna():
    def __init__(self, n_trials, X_train, y_train, X_val, y_val, l1_range, l2_range):

        self.X_train = X_train
        self.y_train = y_train

        self.X_val = X_val
        self.y_val = y_val

        self.n_trials = n_trials
        self.l1_range = l1_range
        self.l2_range = l2_range

    def objective(self, trial):

        self.best_model = None
        self.best_loss = np.inf
        self.best_params = None


        l1 = trial.suggest_float('l1_ratio', l1_range[0], l1_range[1])
        l2 = trial.suggest_float('alpha', l2_range[0], l2_range[1])

        elastic_model = ElasticNet(l1_ratio=l1, alpha=l2)
        
        elastic_model.fit(self.X_train, self.y_train)
        score = root_mean_squared_error(self.y_val, elastic_model.predict(self.X_val))

        if score < self.best_loss:
            self.best_loss = score
            self.best_model = elastic_model
            self.best_params = (l1, l2)
        return score

In [72]:
optimizer = ElasticNetOptuna(100, X_train, y_train, X_val, y_val, [0, 1], [0, 3])
study = optuna.create_study(direction="minimize")
study.optimize(optimizer.objective, n_trials=50)

[I 2025-09-14 05:51:27,616] A new study created in memory with name: no-name-7ad3c446-b30a-4d27-ad03-bba223a75a66
[I 2025-09-14 05:51:27,648] Trial 0 finished with value: 1047.0328388242076 and parameters: {'l1_ratio': 0.1368055487648559, 'alpha': 0.1690697443954175}. Best is trial 0 with value: 1047.0328388242076.
[I 2025-09-14 05:51:27,688] Trial 1 finished with value: 1046.7708199000378 and parameters: {'l1_ratio': 0.12855212064597754, 'alpha': 0.1626470827321902}. Best is trial 1 with value: 1046.7708199000378.
[I 2025-09-14 05:51:27,745] Trial 2 finished with value: 1048.688321311363 and parameters: {'l1_ratio': 0.10154213486802724, 'alpha': 0.1898589039165623}. Best is trial 1 with value: 1046.7708199000378.
[I 2025-09-14 05:51:27,798] Trial 3 finished with value: 1050.4369790834521 and parameters: {'l1_ratio': 0.1636831078108535, 'alpha': 0.23183022601672884}. Best is trial 1 with value: 1046.7708199000378.
[I 2025-09-14 05:51:27,849] Trial 4 finished with value: 1047.5174525042

In [73]:
optuna_model = optimizer.best_model

In [74]:
result_R_2, result_RMSE, result_MAE = check_model_metrics(optuna_model, 
                                                          'Elastic Standart Scaler Optuna', 
                                                          X_train, 
                                                          X_test, 
                                                          y_train, 
                                                          y_test,
                                                          X_val, 
                                                          y_val,
                                                          result_R_2, result_RMSE, result_MAE)

## Comprs

In [75]:
display(result_R_2, result_RMSE, result_MAE)

,model,train,val,test
0,Lasso Standart Scaler,0.586885,0.571805,0.571805
1,Lasso Standart Scaler top10,0.581236,0.564320,0.564320
2,Lasso Standart Scaler Perm Imp,0.579924,0.565504,0.565504
3,Lasso Standart Scaler Shap,0.578988,0.563809,0.563809
4,Elastic Standart Scaler Grid,0.586770,0.572057,0.572057
5,Elastic Standart Scaler Random,0.586748,0.572074,0.572074
6,Elastic Standart Scaler Optuna,0.583757,0.570207,0.570207


,model,train,val,test
0,Lasso Standart Scaler,1015.637572,1043.037205,1043.037205
1,Lasso Standart Scaler top10,1022.558048,1052.114534,1052.114534
2,Lasso Standart Scaler Perm Imp,1024.157766,1050.683104,1050.683104
3,Lasso Standart Scaler Shap,1025.298559,1052.731072,1052.731072
4,Elastic Standart Scaler Grid,1015.778641,1042.729879,1042.729879
5,Elastic Standart Scaler Random,1015.805105,1042.709743,1042.709743
6,Elastic Standart Scaler Optuna,1019.474548,1044.981961,1044.981961


,model,train,val,test
0,Lasso Standart Scaler,705.165992,709.243433,709.243433
1,Lasso Standart Scaler top10,711.763983,717.353878,717.353878
2,Lasso Standart Scaler Perm Imp,713.158844,716.185760,716.185760
3,Lasso Standart Scaler Shap,712.924466,716.413476,716.413476
4,Elastic Standart Scaler Grid,704.989788,708.926748,708.926748
5,Elastic Standart Scaler Random,704.978768,708.900410,708.900410
6,Elastic Standart Scaler Optuna,706.360196,710.122564,710.122564
